In [47]:
import pickle as pkl
import torch
import torch.nn as nn
import torch.optim as optim
from nltk.corpus import brown
from nltk import download
from torch.utils.data import DataLoader, Dataset
from collections import Counter
import random
from tqdm import tqdm
import json

torch.manual_seed(42)

In [48]:
import importlib
import enc_dec_lstm
importlib.reload(enc_dec_lstm)
from enc_dec_lstm import Encoder, Decoder, Encoder_Decoder_Model

In [49]:
device = "xpu" if torch.xpu.is_available() else "cpu"
device

'xpu'

In [50]:
with open('../data/train_data.pkl', 'rb') as f:
    train_data = pkl.load(f)

with open('../data/val_data.pkl', 'rb') as f:
    val_data = pkl.load(f)

In [51]:
word_counts = Counter(w for sent in train_data for w, _ in sent)
tag_counts = Counter(t for sent in train_data for _, t in sent)

word2idx = {w: i+2 for i, (w, _) in enumerate(word_counts.items()) if word_counts[w] > 5}
word2idx["<PAD>"] = 0
word2idx["<UNK>"] = 1
idx2word = {i: w for w, i in word2idx.items()}

tag2idx = {t: i+2 for i, (t, _) in enumerate(tag_counts.items())}
tag2idx["<PAD>"] = 0
tag2idx["<SOS>"] = 1
idx2tag = {i: t for t, i in tag2idx.items()}

In [52]:
with open("tokenizer/word2idx.json", "w") as f:
    json.dump(word2idx, f)
with open("tokenizer/tag2idx.json", "w") as f:
    json.dump(tag2idx, f)

In [44]:
idx2tag

{2: 'PRON',
 3: 'VERB',
 4: 'PRT',
 5: 'ADP',
 6: 'DET',
 7: 'NOUN',
 8: '.',
 9: 'ADV',
 10: 'CONJ',
 11: 'ADJ',
 12: 'NUM',
 13: 'X',
 0: '<PAD>',
 1: '<SOS>'}

In [31]:
vocab_size = len(word2idx)
tag_size = len(tag2idx)

In [ ]:
class POSTagDataset(Dataset):
    def __init__(self, sentences):
        self.data = []
        for sent in sentences:
            words, tags = zip(*sent)
            
            word_ids = [word2idx.get(w, 1) for w in words]
            tag_ids = [tag2idx.get(t, 0) if t in tag2idx else 0 for t in tags]

            length = len(word_ids)

            self.data.append((torch.tensor(word_ids).to(device), torch.tensor(tag_ids).to(device), length))
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]

In [34]:
def data_collate_fn(batch):
    words, tags, lens = zip(*batch)
    words_batch = nn.utils.rnn.pad_sequence(words, batch_first=True, padding_value=0).to(device)
    tags_batch = nn.utils.rnn.pad_sequence(tags, batch_first=True, padding_value=0).to(device)
    return words_batch, tags_batch, lens

In [35]:
train_dataset = POSTagDataset(train_data)
val_dataset = POSTagDataset(val_data)

In [36]:
batch_size=128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=data_collate_fn)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=data_collate_fn)

In [ ]:
epochs = 25
embed_dim = 64
hidden_dim = 128
model=Encoder_Decoder_Model(vocab_size, embed_dim, hidden_dim, tag_size, tag2idx["<SOS>"]).to(device)
lr=0.001
optimizer = optim.Adam(model.parameters(), lr=lr)
# criterion = nn.CrossEntropyLoss()
criterion = nn.CrossEntropyLoss(ignore_index=0)

for e in range(epochs):
    train_loss = 0.0
    for batch in tqdm(train_loader, desc=f"Epoch {e+1}/{epochs}", total=len(train_loader)):
        words_batch, tags_batch, length_batch = batch
        input_seq = words_batch
        output_tags = tags_batch
        outputs = model(input_seq, output_tags, length_batch)
        pred_logits = outputs[0]
        loss = criterion(pred_logits.view(-1, tag_size), output_tags.view(-1))
        train_loss += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"Training Loss: {train_loss / len(train_loader)}")

    with torch.no_grad():
        val_loss = 0.0
        for batch in tqdm(val_loader, desc=f"Validation {e+1}/{epochs}", total=len(val_loader)):
            words_batch, tags_batch, length_batch = batch
            input_seq = words_batch
            output_tags = tags_batch
            outputs = model(input_seq, output_tags, length_batch)
            pred_logits = outputs[0]
            loss = criterion(pred_logits.view(-1, tag_size), output_tags.view(-1))
            val_loss += loss.item()
        print(f"Validation Loss: {val_loss / len(val_loader)}")

    model_path = f'models/encoder_decoder_model_{lr}lr_{batch_size}bs_{e+1}epochs.pth'
    torch.save(model.state_dict(), model_path)

Epoch 1/25:   3%|▎         | 12/359 [00:03<01:32,  3.74it/s]


KeyboardInterrupt: 